In [9]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
from typing import List, Dict, Any
from itertools import product
from tqdm import tqdm

# Variables

In [10]:
# Basic Bus Parameters
bb_stop_time_list = [30] # seconds
bb_hub_stop_time_list = [60, 120, 180, 240, 300] # seconds
bb_transfer_stop_time_list = [60, 90, 120] # seconds

# Ring Bus Parameters
rb_stop_time_list = [30] # seconds
rb_transfer_stop_time_list = [60, 90, 120, 150, 180, 210, 240, 270, 300] # seconds

# The train will arrive at the station at 55min and depart at 60min
TRAIN_ARRIVAL_TIME = 3300
TRAIN_DEPARTURE_TIME = 3600

# Transfer time
TRANSFER_TIME = 30 # seconds

# Bus headway time
BUS_HEADWAY_TIME = 600 # seconds

# Create Basic Bus Schedule

In [11]:
def create_basic_bus_schedule(
    bb_stop_time, bb_hub_stop_time, bb_transfer_stop_time, 
    TRAIN_ARRIVAL_TIME = TRAIN_ARRIVAL_TIME, TRAIN_DEPARTURE_TIME = TRAIN_DEPARTURE_TIME,
):
    bb_hub_arrival_time = (TRAIN_ARRIVAL_TIME + TRAIN_DEPARTURE_TIME) / 2 - bb_hub_stop_time / 2
    bb_hub_departure_time = bb_hub_arrival_time + bb_hub_stop_time

    bb_we_56_departure_time = bb_hub_arrival_time - 3 * 90 - 2 * bb_stop_time
    bb_we_56_arrival_time = bb_we_56_departure_time - bb_transfer_stop_time

    bb_we_59_arrival_time = bb_hub_departure_time + 3 * 90 + 2 * bb_stop_time
    bb_we_59_departure_time = bb_we_59_arrival_time + bb_transfer_stop_time

    bb_ew_56_departure_time = bb_we_59_departure_time
    bb_ew_56_arrival_time = bb_we_59_arrival_time

    bb_ew_59_arrival_time = bb_we_56_arrival_time
    bb_ew_59_departure_time = bb_we_56_departure_time

    bb_ns_88_arrival_time = bb_we_56_arrival_time
    bb_ns_88_departure_time = bb_we_56_departure_time

    bb_ns_27_departure_time = bb_we_59_departure_time
    bb_ns_27_arrival_time = bb_we_59_arrival_time

    bb_sn_27_arrival_time = bb_ew_59_arrival_time
    bb_sn_27_departure_time = bb_ew_59_departure_time

    bb_sn_88_departure_time = bb_ew_56_departure_time
    bb_sn_88_arrival_time = bb_ew_56_arrival_time

    basic_bus_schedule = {
        'BB_WE_56': {
            'departure_time': bb_we_56_departure_time,
            'arrival_time': bb_we_56_arrival_time,
            'departure_time_in_hhmmss': str(timedelta(seconds=bb_we_56_departure_time)),
            'arrival_time_in_hhmmss': str(timedelta(seconds=bb_we_56_arrival_time))
        },
        'BB_WE_59': {
            'departure_time': bb_we_59_departure_time,
            'arrival_time': bb_we_59_arrival_time,
            'departure_time_in_hhmmss': str(timedelta(seconds=bb_we_59_departure_time)),
            'arrival_time_in_hhmmss': str(timedelta(seconds=bb_we_59_arrival_time))
        },
        'BB_EW_56': {
            'departure_time': bb_ew_56_departure_time,
            'arrival_time': bb_ew_56_arrival_time,
            'departure_time_in_hhmmss': str(timedelta(seconds=bb_ew_56_departure_time)),
            'arrival_time_in_hhmmss': str(timedelta(seconds=bb_ew_56_arrival_time))
        },
        'BB_EW_59': {
            'departure_time': bb_ew_59_departure_time,
            'arrival_time': bb_ew_59_arrival_time,
            'departure_time_in_hhmmss': str(timedelta(seconds=bb_ew_59_departure_time)),
            'arrival_time_in_hhmmss': str(timedelta(seconds=bb_ew_59_arrival_time))
        },
        'BB_NS_88': {
            'departure_time': bb_ns_88_departure_time,
            'arrival_time': bb_ns_88_arrival_time,
            'departure_time_in_hhmmss': str(timedelta(seconds=bb_ns_88_departure_time)),
            'arrival_time_in_hhmmss': str(timedelta(seconds=bb_ns_88_arrival_time))
        },
        'BB_NS_27': {
            'departure_time': bb_ns_27_departure_time,
            'arrival_time': bb_ns_27_arrival_time,
            'departure_time_in_hhmmss': str(timedelta(seconds=bb_ns_27_departure_time)),
            'arrival_time_in_hhmmss': str(timedelta(seconds=bb_ns_27_arrival_time))
        },
        'BB_SN_27': {
            'departure_time': bb_sn_27_departure_time,
            'arrival_time': bb_sn_27_arrival_time,
            'departure_time_in_hhmmss': str(timedelta(seconds=bb_sn_27_departure_time)),
            'arrival_time_in_hhmmss': str(timedelta(seconds=bb_sn_27_arrival_time))
        },
        'BB_SN_88': {
            'departure_time': bb_sn_88_departure_time,
            'arrival_time': bb_sn_88_arrival_time,
            'departure_time_in_hhmmss': str(timedelta(seconds=bb_sn_88_departure_time)),
            'arrival_time_in_hhmmss': str(timedelta(seconds=bb_sn_88_arrival_time))
        },
        'HUB': {
            'arrival_time': bb_hub_arrival_time,
            'departure_time': bb_hub_departure_time,
            'arrival_time_in_hhmmss': str(timedelta(seconds=bb_hub_arrival_time)),
            'departure_time_in_hhmmss': str(timedelta(seconds=bb_hub_departure_time))
        }
    }

    basic_station_routes_data = {
        '56': [
            {'route_id': 'BB_WE_0', 'line_id': 'BB_WE', 'base_arrival_time': bb_we_56_arrival_time, 'base_departure_time': bb_we_56_departure_time},
            {'route_id': 'BB_WE_1', 'line_id': 'BB_WE', 'base_arrival_time': bb_ew_56_arrival_time, 'base_departure_time': bb_ew_56_departure_time},
        ],
        '59': [
            {'route_id': 'BB_WE_0', 'line_id': 'BB_WE', 'base_arrival_time': bb_we_59_arrival_time, 'base_departure_time': bb_we_59_departure_time},
            {'route_id': 'BB_WE_1', 'line_id': 'BB_WE', 'base_arrival_time': bb_ew_59_arrival_time, 'base_departure_time': bb_ew_59_departure_time},
        ],
        '27': [
            {'route_id': 'BB_NS_0', 'line_id': 'BB_NS', 'base_arrival_time': bb_ns_27_arrival_time, 'base_departure_time': bb_ns_27_departure_time},
            {'route_id': 'BB_NS_1', 'line_id': 'BB_NS', 'base_arrival_time': bb_sn_27_arrival_time, 'base_departure_time': bb_sn_27_departure_time},
        ],
        '88': [
            {'route_id': 'BB_NS_0', 'line_id': 'BB_NS', 'base_arrival_time': bb_ns_88_arrival_time, 'base_departure_time': bb_ns_88_departure_time},
            {'route_id': 'BB_NS_1', 'line_id': 'BB_NS', 'base_arrival_time': bb_sn_88_arrival_time, 'base_departure_time': bb_sn_88_departure_time},
        ],
    }

    return basic_bus_schedule, basic_station_routes_data



# Create Ring Bus Schedule

In [12]:
def create_ring_bus_schedule(rb_56_departure_time, rb_stop_time, rb_transfer_stop_time):
    rb_0_88_arrival_time = rb_56_departure_time + 6 * 90 + 5 * rb_stop_time
    rb_0_88_departure_time = rb_0_88_arrival_time + rb_transfer_stop_time

    rb_0_59_arrival_time = rb_0_88_departure_time + 6 * 90 + 5 * rb_stop_time
    rb_0_59_departure_time = rb_0_59_arrival_time + rb_transfer_stop_time

    rb_0_27_arrival_time = rb_0_59_departure_time + 6 * 90 + 5 * rb_stop_time
    rb_0_27_departure_time = rb_0_27_arrival_time + rb_transfer_stop_time

    rb_0_56_arrival_time = rb_0_27_departure_time + 6 * 90 + 5 * rb_stop_time

    rb_1_27_arrival_time = rb_0_88_arrival_time
    rb_1_27_departure_time = rb_0_88_departure_time

    rb_1_59_arrival_time = rb_0_59_arrival_time
    rb_1_59_departure_time = rb_0_59_departure_time

    rb_1_88_arrival_time = rb_0_27_arrival_time
    rb_1_88_departure_time = rb_0_27_departure_time

    rb_1_56_arrival_time = rb_0_56_arrival_time

    ring_bus_schedule = {
        'RB_0_56': {
            'arrival_time': rb_0_56_arrival_time,
            'departure_time': rb_56_departure_time,
            'arrival_time_in_hhmmss': str(timedelta(seconds=rb_0_56_arrival_time)),
            'departure_time_in_hhmmss': str(timedelta(seconds=rb_56_departure_time))
        },
        'RB_0_88': {
            'arrival_time': rb_0_88_arrival_time,
            'departure_time': rb_0_88_departure_time,           
            'arrival_time_in_hhmmss': str(timedelta(seconds=rb_0_88_arrival_time)),
            'departure_time_in_hhmmss': str(timedelta(seconds=rb_0_88_departure_time))
        },              
        'RB_0_59': {
            'arrival_time': rb_0_59_arrival_time,
            'departure_time': rb_0_59_departure_time,           
            'arrival_time_in_hhmmss': str(timedelta(seconds=rb_0_59_arrival_time)),
            'departure_time_in_hhmmss': str(timedelta(seconds=rb_0_59_departure_time))
        },
        'RB_0_27': {
            'arrival_time': rb_0_27_arrival_time,
            'departure_time': rb_0_27_departure_time,           
            'arrival_time_in_hhmmss': str(timedelta(seconds=rb_0_27_arrival_time)),
            'departure_time_in_hhmmss': str(timedelta(seconds=rb_0_27_departure_time))
        },
        'RB_1_56': {
            'arrival_time': rb_1_56_arrival_time,
            'departure_time': rb_56_departure_time,
            'arrival_time_in_hhmmss': str(timedelta(seconds=rb_1_56_arrival_time)),
            'departure_time_in_hhmmss': str(timedelta(seconds=rb_56_departure_time))
        },
        'RB_1_27': {
            'arrival_time': rb_1_27_arrival_time,
            'departure_time': rb_1_27_departure_time,           
            'arrival_time_in_hhmmss': str(timedelta(seconds=rb_1_27_arrival_time)),
            'departure_time_in_hhmmss': str(timedelta(seconds=rb_1_27_departure_time))
        },
        'RB_1_59': {
            'arrival_time': rb_1_59_arrival_time,
            'departure_time': rb_1_59_departure_time,           
            'arrival_time_in_hhmmss': str(timedelta(seconds=rb_1_59_arrival_time)),
            'departure_time_in_hhmmss': str(timedelta(seconds=rb_1_59_departure_time))
        },
        'RB_1_88': {
            'arrival_time': rb_1_88_arrival_time,
            'departure_time': rb_1_88_departure_time,           
            'arrival_time_in_hhmmss': str(timedelta(seconds=rb_1_88_arrival_time)),
            'departure_time_in_hhmmss': str(timedelta(seconds=rb_1_88_departure_time))
        }
    }

    ring_station_routes_data = {
        '56': [
            {'route_id': 'RB_0', 'line_id': 'RB', 'base_arrival_time': rb_0_56_arrival_time, 'base_departure_time': rb_56_departure_time},
            {'route_id': 'RB_1', 'line_id': 'RB', 'base_arrival_time': rb_1_56_arrival_time, 'base_departure_time': rb_56_departure_time},
        ],
        '88': [
            {'route_id': 'RB_0', 'line_id': 'RB', 'base_arrival_time': rb_0_88_arrival_time, 'base_departure_time': rb_0_88_departure_time},
            {'route_id': 'RB_1', 'line_id': 'RB', 'base_arrival_time': rb_1_88_arrival_time, 'base_departure_time': rb_1_88_departure_time},
        ],
        '59': [
            {'route_id': 'RB_0', 'line_id': 'RB', 'base_arrival_time': rb_0_59_arrival_time, 'base_departure_time': rb_0_59_departure_time},
            {'route_id': 'RB_1', 'line_id': 'RB', 'base_arrival_time': rb_1_59_arrival_time, 'base_departure_time': rb_1_59_departure_time},
        ],
        '27': [
            {'route_id': 'RB_0', 'line_id': 'RB', 'base_arrival_time': rb_0_27_arrival_time, 'base_departure_time': rb_0_27_departure_time},
            {'route_id': 'RB_1', 'line_id': 'RB', 'base_arrival_time': rb_1_27_arrival_time, 'base_departure_time': rb_1_27_departure_time},
        ],
    }

    return ring_bus_schedule, ring_station_routes_data

# Calculate Total Waiting Time

In [13]:
def calculate_total_wait_time_at_station(
    directional_routes: List[Dict[str, Any]],
    min_transfer_time: float,
    uniform_headway: float
) -> float:
    """
    Calculates the sum of minimum waiting times for all valid transfers
    between directional bus routes using matrix operations.

    Args:
        directional_routes: A list of route dictionaries.
            Each dictionary must contain: 
            - 'route_id' (str): A unique identifier (e.g., "Line1-East").
            - 'line_id' (Any): Identifier for the line (e.g., 1, 2, "A").
            - 'base_arrival_time' (float): Base arrival time at the stop.
            - 'base_departure_time' (float): Base departure time at the stop.
        min_transfer_time: The uniform time required to transfer
            between any two platforms (in the same units as times).
        uniform_headway: The uniform headway (frequency) for all lines
            (in the same units as times).

    Returns:
        The sum of all minimum transfer waiting times.
    """

    # 1. Extract data into NumPy arrays
    # k is the total number of directional routes
    k = len(directional_routes)
    if k == 0:
        return 0.0

    # Create arrays (vectors) of size k
    try:
        arrivals = np.array([r['base_arrival_time'] for r in directional_routes])
        departures = np.array([r['base_departure_time'] for r in directional_routes])
        # We need line_ids to exclude same-line transfers
        line_ids = np.array([r['line_id'] for r in directional_routes])
    except KeyError as e:
        print(f"Error: Input data is missing a required key: {e}")
        return 0.0
    except Exception as e:
        print(f"Error processing input data: {e}")
        return 0.0

    # 2. Calculate the 'Ready Time' vector
    # This is the time when a passenger arriving from route 'i'
    # is ready to board route 'j'.
    # Shape: (k,)
    ready_times = arrivals + min_transfer_time

    # 3. Create the Wait Time Matrix (k x k) using broadcasting
    # We want a matrix where M[i, j] = wait_time(from_A[i] -> to_B[j])

    # Reshape vectors to enable broadcasting:
    # ready_times_col -> (k, 1)
    # departures_row  -> (1, k)
    ready_times_col = ready_times[:, np.newaxis]
    departures_row = departures[np.newaxis, :]

    # Calculate the time difference matrix: (departures_row - ready_times_col)
    # M[i, j] = departures[j] - ready_times[i]
    # This is the base time difference for every (i, j) pair.
    time_diff_matrix = departures_row - ready_times_col

    # Apply the modulo operation to get the minimum wait time.
    # This is the core calculation.
    # wait_matrix[i, j] = (departures[j] - ready_times[i]) % uniform_headway
    wait_matrix = time_diff_matrix % uniform_headway

    # 4. Create a Mask for valid transfers
    # We must exclude transfers where A and B are the same line.
    # (e.g., Line1-East -> Line1-West)

    # Reshape line_ids for broadcasting:
    # line_ids_col -> (k, 1)
    # line_ids_row -> (1, k)
    line_ids_col = line_ids[:, np.newaxis]
    line_ids_row = line_ids[np.newaxis, :]

    # Create a boolean matrix (k x k)
    # mask[i, j] is True if line_ids[i] != line_ids[j]
    # This mask defines all valid transfer pairs.
    mask = (line_ids_col != line_ids_row)
    num_valid_pairs = np.sum(mask)

    # 5. Calculate the final sum
    # Apply the mask to the wait_matrix, which filters out all
    # invalid (same-line) transfers.
    # Then, sum all remaining (valid) wait times.
    total_min_wait_time = np.sum(wait_matrix[mask])

    return total_min_wait_time, num_valid_pairs

In [14]:
def evaluate_basic_ring_bus_schedule( basic_station_routes_data, ring_station_routes_data):
    # Merge basic and ring station routes data
    all_station_ids = set(basic_station_routes_data.keys()) | set(ring_station_routes_data.keys())
    merged_data = {
        station_id: basic_station_routes_data.get(station_id, []) + ring_station_routes_data.get(station_id, [])
        for station_id in all_station_ids
    }

    evaluation_results = {}

    total_wait_time = 0.0
    total_valid_pairs = 0

    for station_id, routes in merged_data.items():
        station_wait_time, num_valid_pairs = calculate_total_wait_time_at_station(
            directional_routes=routes,
            min_transfer_time=TRANSFER_TIME,
            uniform_headway=BUS_HEADWAY_TIME
        )
        evaluation_results[f'station_{station_id}_total_wait_time'] = station_wait_time
        station_average_wait_time = station_wait_time / num_valid_pairs if num_valid_pairs > 0 else 0.0
        evaluation_results[f'station_{station_id}_average_wait_time'] = station_average_wait_time
        evaluation_results[f'station_{station_id}_num_valid_pairs'] = num_valid_pairs
        total_wait_time += station_wait_time
        total_valid_pairs += num_valid_pairs
    evaluation_results['total_wait_time_all_stations'] = total_wait_time
    evaluation_results['average_wait_time_stations'] = total_wait_time / len(all_station_ids) if merged_data else 0.0
    evaluation_results['total_valid_pairs'] = total_valid_pairs
    evaluation_results['average_wait_time_lines'] = total_wait_time / total_valid_pairs if total_valid_pairs > 0 else 0.0
    return evaluation_results

In [15]:
all_combinations = list(product(
    bb_stop_time_list,
    bb_hub_stop_time_list,
    bb_transfer_stop_time_list,
    rb_stop_time_list,
    rb_transfer_stop_time_list
))

print(f"Total combinations to evaluate: {len(all_combinations)}")

Total combinations to evaluate: 135


In [16]:
all_evaluation_results = []

for bus_params in tqdm(all_combinations):
    bb_stop_time, bb_hub_stop_time, bb_transfer_stop_time, rb_stop_time, rb_transfer_stop_time = bus_params

    # Create basic bus schedule
    basic_bus_schedule, basic_station_routes_data = create_basic_bus_schedule(
        bb_stop_time, bb_hub_stop_time, bb_transfer_stop_time
    )

    # Determine ring bus departure time at the station 56
    bb_we_56_arrival_time = basic_bus_schedule['BB_WE_56']['arrival_time']
    bb_we_56_arrival_time_next = bb_we_56_arrival_time + BUS_HEADWAY_TIME

    rb_56_departure_time_list = list(range(int(bb_we_56_arrival_time), int(bb_we_56_arrival_time_next) + 1, 30))

    for rb_56_departure_time in rb_56_departure_time_list:
        # Create ring bus schedule
        ring_bus_schedule, ring_station_routes_data = create_ring_bus_schedule(rb_56_departure_time, rb_stop_time, rb_transfer_stop_time)

        # Evaluate the combined schedule
        evaluation_results = evaluate_basic_ring_bus_schedule(basic_station_routes_data, ring_station_routes_data)

        # Add bus parameters to the results
        evaluation_results.update({
            'bb_stop_time': bb_stop_time,
            'bb_hub_stop_time': bb_hub_stop_time,
            'bb_transfer_stop_time': bb_transfer_stop_time,
            'rb_stop_time': rb_stop_time,
            'rb_transfer_stop_time': rb_transfer_stop_time,
            'rb_56_departure_time': rb_56_departure_time
        })
        all_evaluation_results.append(evaluation_results)

# Convert results to DataFrame for easier analysis
results_df = pd.DataFrame(all_evaluation_results)
print(f'Evaluation completed. Total combinations evaluated: {len(results_df)}')

# Save results to CSV
save_dirpath = 'data/GTFS/bus'
os.makedirs(save_dirpath, exist_ok=True)
results_filepath = os.path.join(save_dirpath, 'basic_ring_bus_schedule_evaluation_results.csv')
results_df.to_csv(results_filepath, index=False)
print(f'Results saved to {results_filepath}')

100%|██████████| 135/135 [00:00<00:00, 901.35it/s]

Evaluation completed. Total combinations evaluated: 2835
Results saved to data/GTFS/bus\basic_ring_bus_schedule_evaluation_results.csv
